# CottonLens — sürekli GPU araştırması
T4 GPU seçip ilk beş kod hücresini çalıştır. Son hücre diagnose → search → compare turunu tamamlar.
Bu geniş arama saatler sürebilir; tamamlanan fitler Drive sicilinden devam eder. Eski R2/sprint değişmez.
Varsayılan Tur A: XGBoost ve CatBoost için 128 aday/aile/horizon; iç bloklar ve üç seed doğrulaması.
B: yön; C: sequence; D: genişletilmiş feature; E: history/cadence/ensemble. Tur değiştirmek için yalnız ROUND değerini değiştir.
2024+ seçimde kullanılmaz. Sonuçlar görülmüş tarihsel araştırma kanıtıdır. CatBoost GPU yeniden üretim toleransını geçemezse export durur.
Oturum kesilirse ilk hücrelerden tekrar başla. Aktif oturum varken ikinci yazıcı çalıştırma; stale lock dosyasını ancak önceki oturumun kapandığı doğrulanınca kaldır.
V2: RangeDataset CPU/GPU scope fix. Previous research-v1 evidence is preserved; short controls repeat as bugfix verification under a new immutable identity.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-v2-tf-placement'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-research-source-v2.zip'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected_zip, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO, '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
run([*BASE, '--stage', 'prepare'])


In [ ]:
ROUND = 'A'
EXTENSION = 0  # Keep 0 for the initial round. Extensions require recorded inner-validation improvement.
run([*BASE, '--stage', 'round', '--round', ROUND, '--extension', str(EXTENSION)])


## Aday kilitleme ve release
Arama turlarını değerlendirdikten sonra STAGE seç: lock → reproduce → export. Varsayılan compare hiçbir adayı kilitlemez. Kilitlenen deneyde yeni arama yapılmaz. Export yalnız yeniden üretim ve ayrı CPU runtime kontrolü geçerse ZIP oluşturur. Yeni sonuç eski demo veya DB üzerine kurulmaz.
CFTC/WASDE/kontrat verisi otomatik etkin değildir: gerçek yayın/vintage kanıtı içeren paket ayrı research kimliğiyle prepare aşamasına verilmelidir.

In [ ]:
STAGE = 'compare'  # Later: lock, then reproduce, then export, one at a time.
assert STAGE in ('compare', 'lock', 'reproduce', 'export')
run([*BASE, '--stage', STAGE])
